<a href="https://colab.research.google.com/github/laurindodumba/AULA-FUNDAMENTO-DE-INTELIGENCIA-ARTIFICIAL/blob/main/Aula_06_Chat_PDF_corrigido.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📄 Aula 06 — Chat com PDF

Pipeline RAG usando LangChain

**Fluxo:**
1. Carregar PDF
2. Quebrar em chunks
3. Gerar embeddings
4. Guardar no banco vetorial (Chroma)
5. Retriever busca os trechos relevantes
6. LLM responde com base no contexto

In [ ]:
# Instalação das bibliotecas
!pip install -q \
    langchain-community \
    langchain-text-splitters \
    langchain-huggingface \
    langchain-core \
    chromadb \
    sentence-transformers \
    transformers \
    pypdf \
    huggingface_hub \
    accelerate

In [ ]:
# Imports
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from transformers import pipeline

print("Imports OK!")

Imports OK!


In [ ]:
# 1. Carregar PDF
# Faça upload do seu PDF no Colab antes de executar esta célula
arquivo = '/content/pdf_1772.pdf'

loader = PyPDFLoader(arquivo)
documents = loader.load()



In [ ]:
print(f"Total de páginas carregadas: {len(documents)}")


Total de páginas carregadas: 5


In [ ]:
print("\nPrimeira página (trecho):")



Primeira página (trecho):


In [ ]:
print(documents[0].page_content[200:1000])

atégica sobre a gestão de pessoas, a nossa especialização em Gestão de Pessoas e Psicologia Organizacional e do
Trabalho é a oportunidade que você estava esperando. 
Com uma formação focada em resultados práticos e soluções inovadoras, este curso é ideal para quem quer se destacar no
mercado de trabalho, seja em grandes empresas, startups ou no campo do empreendedorismo.
Destinado a profissionais de diversas áreas, como psicólogos, gestores, pedagogos e administradores, o curso oferece uma visão
abrangente sobre o comportamento organizacional, a cultura corporativa e as ferramentas mais avançadas para otimizar a gestão de
pessoas. Ao longo da especialização, você aprenderá a aplicar o People Analytics para analisar dados comportamentais e de
desempenho, criando soluções mais eficazes e per


In [ ]:
# 2. Quebrar texto em chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=80
)
docs = text_splitter.split_documents(documents)

print(f"Total de chunks gerados: {len(docs)}")

Total de chunks gerados: 57


In [ ]:
# 3. Embeddings + Banco vetorial + Retriever
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

db = Chroma.from_documents(docs, embeddings)

retriever = db.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3}
)

print("Banco vetorial e retriever prontos!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Banco vetorial e retriever prontos!


In [ ]:
# 4. LLM local

pipe = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    max_new_tokens=200,
)

llm = HuggingFacePipeline(pipeline=pipe)

print("Modelo LLM carregado!")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLl

Modelo LLM carregado!


In [ ]:
# 5. Prompt + Chain

prompt = PromptTemplate.from_template("""\
Você é um assistente especializado em análise de documentos.
Responda de forma clara e objetiva com base apenas no contexto abaixo.
Se não encontrar a resposta, diga: "Não encontrei essa informação no documento."

Contexto:
{context}

Pergunta: {question}

Resposta:""")

def formatar_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Pipeline LCEL: recupera → formata → prompt → llm → texto
chain = (
    {"context": retriever | formatar_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("Ok!")

Ok!


In [ ]:
# 6. Loop de perguntas
print("Chat com PDF iniciado! Digite 'sair' para encerrar.\n")

while True:
    pergunta = input("Pergunta: ")
    if pergunta.lower() == "sair":
        print("Encerrando...")
        break

    resposta = chain.invoke(pergunta)
    print("\nResposta:", resposta)
    print("-" * 60)

Chat com PDF iniciado! Digite 'sair' para encerrar.

Pergunta: o que fala este pdf


Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Resposta: Você é um assistente especializado em análise de documentos.
Responda de forma clara e objetiva com base apenas no contexto abaixo.
Se não encontrar a resposta, diga: "Não encontrei essa informação no documento."

Contexto:
práticas e ferramentas para a resolução eficaz de conflitos e a gestão de erros no contexto de gestão de pessoas.
E-mail:
pos.toledo@pucpr.br
Telefone:
45991549135 www.pucpr.br

Conflitos e Resolução de Problemas
Esta disciplina te como objetivo aprofundar o entendimento dos estudantes sobre a natureza dos conflitos
organizacionais e como lidar com erros e conflitos de maneira construtiva. O curso abordará teorias, estratégias

projetar, implementar e avaliar programas de aprendizado que capacitam indivíduos e organizações a prosperar em um
mundo impulsionado pela revolução digital.
Gestão de Pessoas em Cooperativas: Estratégias para o Fortalecimento da Colaboração e
Sucesso Colet

Pergunta: o que fala este pdf

Resposta:
---------------------------------

---
## Groq (gratuito)

1. Crie sua chave gratuita em: https://console.groq.com
2. E teste

In [ ]:
# Testando o GROK
!pip install -q langchain-groq

import os
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Cole aqui sua chave Groq
os.environ["GROQ_API_KEY"] = ""

llm_groq = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

prompt = PromptTemplate.from_template("""\
Você é um assistente especializado em análise de documentos.
Responda de forma clara e objetiva com base apenas no contexto abaixo.
Se não encontrar a resposta, diga: "Não encontrei essa informação no documento."

Contexto:
{context}

Pergunta: {question}

Resposta:""")

def formatar_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

chain = (
    {"context": retriever | formatar_docs, "question": RunnablePassthrough()}
    | prompt
    | llm_groq
    | StrOutputParser()
)

print("Chain com Groq pronta! Digite 'sair' para encerrar.\n")

while True:
    pergunta = input("Pergunta: ")
    if pergunta.lower() == "sair":
        print("Encerrando...")
        break

    resposta = chain.invoke(pergunta)
    print("\nResposta:", resposta)
    print("-" * 60)

Chain com Groq pronta! Digite 'sair' para encerrar.

Pergunta: Olá tudo bem !

Resposta: Sim, tudo bem. Como posso ajudar você hoje? Você tem alguma pergunta sobre o curso de Tech Recruiting ou gostaria de saber mais sobre o assunto?
------------------------------------------------------------
Pergunta: Fala sobre este pdf

Resposta: Este documento parece ser um resumo de uma disciplina acadêmica chamada "Conflitos e Resolução de Problemas". O objetivo da disciplina é aprofundar o entendimento dos estudantes sobre conflitos organizacionais e como lidar com erros e conflitos de maneira construtiva.

A disciplina aborda teorias, estratégias e ferramentas práticas para melhorar a comunicação, promover a cultura colaborativa e otimizar o desempenho individual e coletivo nas organizações. O objetivo é formar profissionais capazes de liderar equipes de maneira eficaz.

O documento também fornece informações de contato, incluindo um endereço de e-mail (pos.toledo@pucpr.br) e um telefone (4599